## 만들어야 하는 llm node 목록

- fall_case_node
- subsequence_router_node
- product_classification_node
- intent_router_node
- answer_with_result_node

In [3]:
!pip install -q python-dotenv pydantic langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 16.3 MB/s eta 0:00:00


In [1]:
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

# from google.colab import userdata
# import os

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

True

In [2]:
def llm_test(llm_prompt, struct=None):
    if struct is not None:
        buff = ChatOpenAI(model="gpt-4o-mini")
        model = buff.with_structured_output(struct)
    else:
        model = ChatOpenAI(model="gpt-4o-mini")

    chatlist = []
    chatlist.append(SystemMessage(content=llm_prompt))

    while True:
        user_input = input("사용자 입력: ")
        if user_input == "q":
            break

        print("사용자 입력:", user_input)

        handler = []

        if struct is None:
            chatlist.append(HumanMessage(content=user_input))
            handler = [c for c in chatlist]
        else:
            handler.append(chatlist[0])
            handler.append(HumanMessage(content=user_input))

        response = model.invoke(handler)

        print(response)

        if struct is None:
            chatlist.append(AIMessage(content=response.content))

def matrix(correct_value: list[bool], predict_value: list[bool]):
    """
    confusion matrix 및 accuracy 출력

    Parameters
    ----------
    correct_value : list[bool]
        실제 정답값 리스트

    predict_value : list[bool]
        예측값 리스트
    """

    if len(correct_value) != len(predict_value):
        raise ValueError("두 리스트의 길이는 같아야 합니다.")

    tp = tn = fp = fn = 0

    for correct, predict in zip(correct_value, predict_value):

        # True를 True로 예측
        if correct is True and predict is True:
            tp += 1

        # False를 False로 예측
        elif correct is False and predict is False:
            tn += 1

        # False를 True로 예측
        elif correct is False and predict is True:
            fp += 1

        # True를 False로 예측
        elif correct is True and predict is False:
            fn += 1

    total = tp + tn + fp + fn

    accuracy = (tp + tn) / total if total > 0 else 0

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    print("Confusion Matrix")
    print()
    print("                Predicted")
    print("              Positive   Negative")
    print(f"Actual Positive   {tp:^5}      {fn:^5}")
    print(f"Actual Negative   {fp:^5}      {tn:^5}")

    print()
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

In [7]:
# 구조를 제공하고 싶은 경우

# 답변으로 받을 구조를 설정
class SubsequenceStructure(BaseModel):
    is_subsequence: bool

# 여기서 프롬프트를 설정하고 입력
prompt = "사용자의 입력을 받고 사용자의 의도를 파악해라, ~ 중에 ~한 상품 있어? 라고 물어보면 subsequence 한 질문으로 판단해 True를 반환해라"

# 테스트는 이렇게
llm_test(prompt, SubsequenceStructure)

사용자 입력: 내일 비올까?
사용자 입력: 내일 비올까?
is_subsequence=False
사용자 입력: 너 임마 그렇게 살면 안돼
사용자 입력: 너 임마 그렇게 살면 안돼
is_subsequence=False
사용자 입력: 너 임마 그렇게 살면 안되는 상품있어?
사용자 입력: 너 임마 그렇게 살면 안되는 상품있어?
is_subsequence=False
사용자 입력: TV 
사용자 입력: TV 
is_subsequence=False
사용자 입력: 만원 짜리 상품 있어?
사용자 입력: 만원 짜리 상품 있어?
is_subsequence=False
사용자 입력: 방금 전 상품 중에 만원 짜리 있어?
사용자 입력: 방금 전 상품 중에 만원 짜리 있어?
is_subsequence=True


KeyboardInterrupt: Interrupted by user

In [8]:
# 멀티턴 대화

# 여기서 프롬프트를 설정하고 입력
prompt = "너는 친절한 챗봇이야"

# 테스트는 이렇게
llm_test(prompt)

사용자 입력: 안녕
사용자 입력: 안녕
content='안녕하세요! 어떻게 도와드릴까요?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 24, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_dc5460657f', 'id': 'chatcmpl-DffvLZcVw9bjcfKUKoGlV3gXCspnb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e2a3b-1231-7442-9207-d4f4a68db9e0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 24, 'output_tokens': 10, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
사용자 입력: TV를 골라야해
사용자 입력: TV를 골라야해
content='TV를 고르는 데 도움이 필요하신가요? 어떤 용도로 사용하실 건지, 어떤 크기를 원하시는지, 예산은 얼마인지 말씀해 

KeyboardInterrupt: Interrupted by user

In [9]:
# 테스트
true_values = [True, True, False, False, True]
predict_values = [False, True, False, False, True]
matrix(true_values, predict_values)

Confusion Matrix

                Predicted
              Positive   Negative
Actual Positive     3          2  
Actual Negative     2          3  

Accuracy : 0.6000
Precision: 0.6000
Recall   : 0.6000
F1 Score : 0.6000


In [5]:
# =========================
# LLM 설정
# =========================

import os
# from google.colab import userdata
from langchain_openai import ChatOpenAI

# # Colab Secrets에서 OpenAI API Key 불러오기
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# LLM 객체 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [4]:
# =========================
# 구조화 출력 스키마
# =========================

from pydantic import BaseModel, Field

class FallCaseStructure(BaseModel):
    is_fall_case: bool = Field(
        description="사용자의 입력이 전자제품 상담 범위를 벗어난 fall case이면 True, 아니면 False"
    )
    response: str = Field(
        description="fall case일 때 사용자에게 반환할 안내 응답. fall case가 아니면 빈 문자열"
    )

In [9]:
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate

# =========================
# fall_case_node 50개 테스트
# =========================

# 0. fall_case_prompt 정의
fall_case_prompt = """
너는 전자제품 상담 챗봇의 fall case 분류기다.

너의 역할은 사용자의 입력이 전자제품 상담 챗봇의 처리 범위에 해당하는지 판단하는 것이다.

전자제품 상담 챗봇의 처리 범위는 다음과 같다.
- 에어컨, TV, 냉장고, 청소기, 세탁기 등 전자제품 관련 질문
- 제품 추천 요청
- 가격, 용량, 크기, 색상, 성능, 에너지 등급 등 조건 기반 제품 검색
- 제품 사용법, 기능, 설명서, 오류 해결 관련 질문
- 이전 전자제품 상담 대화에 이어지는 후속 질문

다음에 해당하면 is_fall_case=True로 판단한다.
- 전자제품과 무관한 질문
- 정치, 연애, 음식, 여행, 날씨, 일반 지식 등 챗봇 목적과 무관한 질문
- 단순 잡담
- 욕설만 있는 입력
- 의미를 판단하기 어려운 입력
- 제품 상담과 관련 없는 요청

다음에 해당하면 is_fall_case=False로 판단한다.
- 전자제품 제품군, 제품명, 기능, 사용법, 조건 검색, 추천과 관련된 질문
- 문장이 짧거나 다소 모호하더라도 전자제품 상담 맥락으로 해석 가능한 질문
- 이전 대화의 제품 상담 흐름을 이어가는 후속 질문

출력 규칙:
- is_fall_case가 True이면 response에 짧고 자연스럽게 전자제품 상담으로 유도하는 답변을 작성한다.
- is_fall_case가 False이면 response는 빈 문자열("")로 둔다.
"""

# 1. 테스트 데이터 로드
test_df = pd.read_csv("fall_case_label_test_50.csv")

# 컬럼명 확인
print(test_df.columns)
display(test_df.head())

# 2. fall_case 테스트용 체인 생성
fall_case_test_prompt = ChatPromptTemplate.from_messages([
    ("system", fall_case_prompt),
    ("human", "{user_input}")
])

fall_case_test_chain = fall_case_test_prompt | llm.with_structured_output(FallCaseStructure)

# 3. 예측 실행
predict_values = []
responses = []

for question in test_df["사용자의 질문"]:
    result = fall_case_test_chain.invoke({
        "user_input": question
    })

    predict_values.append(result.is_fall_case)
    responses.append(result.response)

test_df["predict_label"] = predict_values
test_df["response"] = responses

display(test_df.head())

# 4. 정답 라벨 정리
# CSV에서 True/False가 문자열로 읽히는 경우까지 대응
def to_bool(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        value = value.strip().lower()

        if value == "true":
            return True
        elif value == "false":
            return False
        else:
            raise ValueError(f"bool로 변환할 수 없는 문자열입니다: {value}")

    return bool(value)

correct_values = test_df["정답 라벨"].apply(to_bool).tolist()
predict_values = test_df["predict_label"].apply(to_bool).tolist()

# 5. 기존 matrix 함수로 평가
matrix(correct_values, predict_values)

# 6. 오답 케이스 확인
wrong_df = test_df[
    test_df["정답 라벨"].apply(to_bool) != test_df["predict_label"].apply(to_bool)
]

display(wrong_df)

# 7. 테스트 결과 저장
test_df.to_csv("fall_case_test_result.csv", index=False, encoding="utf-8-sig")

print("저장 완료: fall_case_test_result.csv")

Index(['사용자의 질문', '정답 라벨'], dtype='object')


,사용자의 질문,정답 라벨
0,내 TV에서 화면 밝기 조절 어떻게 해?,False
1,내가 등록한 냉장고 필터 교체 방법 찾아줘,False
2,세탁기 에러코드 UE가 무슨 뜻이야?,False
3,자동 건조 기능 있는 세탁기 찾아줘,False
4,저소음 모드가 있는 청소기 있어?,False


,사용자의 질문,정답 라벨,predict_label,response
0,내 TV에서 화면 밝기 조절 어떻게 해?,False,False,
1,내가 등록한 냉장고 필터 교체 방법 찾아줘,False,False,
2,세탁기 에러코드 UE가 무슨 뜻이야?,False,False,
3,자동 건조 기능 있는 세탁기 찾아줘,False,False,
4,저소음 모드가 있는 청소기 있어?,False,False,


Confusion Matrix

                Predicted
              Positive   Negative
Actual Positive    24          1  
Actual Negative     3         22  

Accuracy : 0.9200
Precision: 0.8889
Recall   : 0.9600
F1 Score : 0.9231


,사용자의 질문,정답 라벨,predict_label,response
16,비교해줘,False,True,어떤 전자제품을 비교해드릴까요? 제품명이나 종류를 알려주시면 도움을 드리겠습니다.
17,안녕,False,True,안녕하세요! 전자제품에 대한 질문이 있으시면 도와드릴게요.
18,질문해도 돼?,False,True,전자제품에 대한 질문이 있으시면 말씀해 주세요!
39,이 TV 바로 구매해줘,True,False,


저장 완료: fall_case_test_result.csv


In [7]:
# =========================
# subsequence_router_node
# =========================

from pydantic import BaseModel, Field


class SubsequenceRouterStructure(BaseModel):
    is_subsequence: bool = Field(
        description="사용자 입력이 이전 대화나 직전 추천/검색 결과를 이어받는 후속 질문이면 True, 독립적인 새 질문이면 False"
    )


subsequence_router_prompt = """
너는 전자제품 사용설명서 기반 Q&A 및 제품 검색 챗봇의 후속 질문 판별기이다.

너의 역할은 사용자의 현재 입력이 이전 대화 맥락을 이어받는 후속 질문인지,
아니면 독립적으로 처리 가능한 새로운 질문인지 판단하는 것이다.

후속 질문이란, 현재 입력만으로는 의미가 완전히 확정되지 않고
이전 대화에서 언급된 제품, 조건, 추천 결과, 비교 대상, 검색 결과를 참고해야 하는 입력을 말한다.

다음과 같은 경우 is_subsequence를 True로 판단한다.

1. 이전 추천 결과나 검색 결과를 이어받는 질문
예시:
- 그중에 제일 싼 거 보여줘
- 아까 말한 제품 더 설명해줘
- 두 번째 거는 어때?
- 그 제품 단점은 뭐야?
- 방금 추천한 것 중에 에너지 효율 좋은 거 있어?

2. 이전에 언급된 조건을 유지한 채 추가 조건을 붙이는 질문
예시:
- 그럼 100만원 이하로만 보여줘
- 용량은 더 큰 걸로
- 소음 적은 걸로 다시 찾아줘
- 디자인 괜찮은 것도 있어?
- 설치 쉬운 제품으로 다시 추천해줘

3. 지시어가 포함되어 이전 맥락이 필요한 질문
예시:
- 이거 설명해줘
- 저거 말고 다른 거
- 그건 제외하고
- 아까 거랑 비교해줘
- 비슷한 제품 있어?

4. 제품명, 카테고리, 조건이 생략되어 단독으로 의미가 불완전한 질문
예시:
- 더 큰 건?
- 더 싼 건?
- 비교해줘
- 다시 추천해줘
- 장점은?
- 단점은?
- 사용법은?

다음과 같은 경우 is_subsequence를 False로 판단한다.

1. 현재 입력만으로 제품 카테고리와 요청 의도가 명확한 새 질문
예시:
- 100만원 이하 TV 추천해줘
- 냉장고 필터 교체 방법 알려줘
- 에어컨 자동 청소 기능 있는 제품 찾아줘
- 세탁기 에러코드 UE가 뭐야?
- 청소기 흡입력 좋은 제품 추천해줘

2. 새로운 제품 카테고리나 새로운 제품명을 명확히 제시한 질문
예시:
- 이번에는 냉장고 찾아줘
- LG TV 사용법 알려줘
- 삼성 세탁기 건조 기능 설명해줘
- 에어컨 전기세 적게 나오는 모델 있어?

3. 단순 인사나 상담 시작 표현
예시:
- 안녕
- 질문해도 돼?
- 제품 좀 찾아줘
- 사용설명서 검색하고 싶어

판단 기준:
- 이전 대화가 있어야만 정확히 이해할 수 있으면 True이다.
- 현재 입력 하나만으로도 충분히 처리 가능하면 False이다.
- "그거", "이거", "저거", "아까", "방금", "그중", "두 번째", "다른 거", "더" 같은 표현이 있고 이전 맥락을 참조하면 True이다.
- 단, 제품 카테고리와 요청 조건이 현재 입력에 명확히 포함되어 있으면 False이다.
- 애매하면 True로 판단한다. 후속 질문으로 보내는 것이 더 안전하다.

출력 규칙:
- 반드시 is_subsequence 값만 판단한다.
- 후속 질문이면 True
- 새로운 질문이면 False
"""


# 테스트
llm_test(subsequence_router_prompt, SubsequenceRouterStructure)

In [8]:
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate

# =========================
# subsequence_router_node 테스트
# =========================

# 1. 테스트 데이터 로드
test_df = pd.read_csv("subsequence_router_label_test_50.csv")

# 컬럼명 확인
print(test_df.columns)
display(test_df.head())

# 2. 테스트용 프롬프트 생성
subsequence_router_test_prompt = ChatPromptTemplate.from_messages([
    ("system", subsequence_router_prompt),
    ("human", """
현재 사용자 입력:
{user_input}
""")
])

# 3. 테스트용 체인 생성
subsequence_router_test_chain = (
    subsequence_router_test_prompt
    | llm.with_structured_output(SubsequenceRouterStructure)
)

# 4. 예측 실행
predict_values = []

for _, row in test_df.iterrows():
    result = subsequence_router_test_chain.invoke({
        "user_input": row["사용자의 질문"]
    })

    predict_values.append(result.is_subsequence)

test_df["predict_label"] = predict_values

display(test_df.head())

# 5. 정답 라벨 정리
def to_bool(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        value = value.strip().lower()

        if value == "true":
            return True
        elif value == "false":
            return False
        else:
            raise ValueError(f"bool로 변환할 수 없는 문자열입니다: {value}")

    return bool(value)

correct_values = test_df["정답 라벨"].apply(to_bool).tolist()
predict_values = test_df["predict_label"].apply(to_bool).tolist()

# 6. 기존 matrix 함수로 평가
matrix(correct_values, predict_values)

# 7. 오답 케이스 확인
wrong_df = test_df[
    test_df["정답 라벨"].apply(to_bool) != test_df["predict_label"].apply(to_bool)
]

display(wrong_df)

# 8. 테스트 결과 저장
test_df.to_csv("subsequence_router_test_result.csv", index=False, encoding="utf-8-sig")

print("저장 완료: subsequence_router_test_result.csv")

Index(['사용자의 질문', '정답 라벨'], dtype='object')


,사용자의 질문,정답 라벨
0,그중에 제일 싼 거 보여줘,True
1,아까 말한 제품 더 설명해줘,True
2,두 번째 거는 어때?,True
3,그 제품 단점은 뭐야?,True
4,방금 추천한 것 중에 에너지 효율 좋은 거 있어?,True


,사용자의 질문,정답 라벨,predict_label
0,그중에 제일 싼 거 보여줘,True,True
1,아까 말한 제품 더 설명해줘,True,True
2,두 번째 거는 어때?,True,True
3,그 제품 단점은 뭐야?,True,True
4,방금 추천한 것 중에 에너지 효율 좋은 거 있어?,True,True


Confusion Matrix

                Predicted
              Positive   Negative
Actual Positive    25          0  
Actual Negative     0         25  

Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000


,사용자의 질문,정답 라벨,predict_label


저장 완료: subsequence_router_test_result.csv


In [9]:
# =========================
# product_classification_node
# =========================

from pydantic import BaseModel, Field


class ProductClassificationStructure(BaseModel):
    product_category: str = Field(
        description='사용자 입력에서 식별된 제품 카테고리 코드. TV는 "TVT", 에어컨은 "ACT", 냉장고는 "REF", 청소기는 "VAC", 세탁기는 "WMT". 제품 의도가 없거나 판단 불가능하면 빈 문자열 ""'
    )


product_classification_prompt = """
너는 전자제품 사용설명서 기반 Q&A 및 제품 검색 챗봇의 제품 카테고리 분류기이다.

너의 역할은 사용자 입력에서 어떤 제품 카테고리를 말하고 있는지 판단하여,
정해진 제품 카테고리 코드 하나를 반환하는 것이다.

반환 가능한 제품 카테고리 코드는 다음과 같다.

- TV: "TVT"
- 에어컨: "ACT"
- 냉장고: "REF"
- 청소기: "VAC"
- 세탁기: "WMT"

제품 카테고리 판단 기준은 다음과 같다.

1. TV로 분류하는 경우 → "TVT"
다음 표현이 포함되면 TV로 판단한다.
- TV
- 티비
- 텔레비전
- 스마트 TV
- 모니터처럼 쓰는 TV
- 화면 밝기, 채널, 리모컨, 셋톱박스 연결 등 TV 사용 맥락

예시:
- TV 화면이 너무 어두워
- 티비 리모컨 연결 방법 알려줘
- 100만원 이하 스마트 TV 찾아줘
- 넷플릭스 되는 TV 있어?
출력:
product_category = "TVT"

2. 에어컨으로 분류하는 경우 → "ACT"
다음 표현이 포함되면 에어컨으로 판단한다.
- 에어컨
- 에어컨디셔너
- 냉방
- 난방
- 제습
- 송풍
- 실외기
- 자동 청소
- 무풍
- 벽걸이형, 스탠드형 에어컨

예시:
- 에어컨 자동 청소 기능 어떻게 써?
- 제습 잘 되는 에어컨 찾아줘
- 벽걸이 에어컨 추천해줘
- 에어컨 전기세 적게 쓰는 방법 알려줘
출력:
product_category = "ACT"

3. 냉장고로 분류하는 경우 → "REF"
다음 표현이 포함되면 냉장고로 판단한다.
- 냉장고
- 김치냉장고
- 냉동고
- 냉장실
- 냉동실
- 정수 필터
- 아이스메이커
- 신선 보관
- 양문형, 4도어 냉장고

예시:
- 냉장고 필터 교체 방법 알려줘
- 500L 이상 냉장고 찾아줘
- 냉동실 온도 조절 어떻게 해?
- 김치냉장고도 검색 가능해?
출력:
product_category = "REF"

4. 청소기로 분류하는 경우 → "VAC"
다음 표현이 포함되면 청소기로 판단한다.
- 청소기
- 무선청소기
- 유선청소기
- 로봇청소기
- 흡입력
- 먼지통
- 브러시
- 물걸레
- 배터리
- 충전 거치대

예시:
- 흡입력 좋은 청소기 찾아줘
- 청소기 먼지통 비우는 법 알려줘
- 로봇청소기 물걸레 기능 있어?
- 무선청소기 배터리 오래 가는 제품 추천해줘
출력:
product_category = "VAC"

5. 세탁기로 분류하는 경우 → "WMT"
다음 표현이 포함되면 세탁기로 판단한다.
- 세탁기
- 드럼세탁기
- 통돌이
- 세탁
- 탈수
- 헹굼
- 건조
- 세제함
- 배수
- 세탁 코스
- 에러코드 UE, OE, IE 등 세탁기 오류 맥락

예시:
- 세탁기 UE 에러가 뭐야?
- 드럼세탁기 청소 방법 알려줘
- 건조 기능 있는 세탁기 찾아줘
- 탈수가 안 돼
출력:
product_category = "WMT"

빈 문자열 ""로 반환하는 경우는 다음과 같다.

1. 제품 카테고리 의도가 없는 경우
예시:
- 안녕
- 오늘 날씨 어때?
- 점심 뭐 먹지?
- 파이썬 코드 짜줘
- 영화 추천해줘

출력:
product_category = ""

2. 전자제품과 관련은 있어 보이지만 지원 카테고리에 없는 경우
예시:
- 전자레인지 추천해줘
- 노트북 찾아줘
- 공기청정기 필터 교체 방법 알려줘
- 밥솥 사용법 알려줘
- 선풍기 추천해줘

출력:
product_category = ""

3. 제품명이 생략되어 현재 입력만으로 카테고리를 판단할 수 없는 후속 질문
예시:
- 그중에 제일 싼 거
- 두 번째 거 설명해줘
- 더 큰 걸로 보여줘
- 사용법 알려줘
- 비교해줘

출력:
product_category = ""

주의사항:
- 반드시 위 5개 코드 중 하나 또는 빈 문자열 ""만 반환한다.
- 여러 제품이 동시에 언급되면 가장 중심이 되는 제품 하나만 선택한다.
- 사용자가 명확히 비교를 요청하며 여러 제품을 동등하게 언급한 경우에도 하나로 확정하기 어렵다면 빈 문자열 ""을 반환한다.
- 제품 카테고리보다 브랜드명만 있는 경우에는 빈 문자열 ""을 반환한다.
  예시: "삼성 거 추천해줘", "LG 제품 찾아줘" → ""
- "건조기"는 세탁기와 다르므로 단독으로 나오면 빈 문자열 ""을 반환한다.
- "김치냉장고"는 냉장고 범주로 보고 "REF"를 반환한다.
- "로봇청소기"는 청소기 범주로 보고 "VAC"를 반환한다.
- 현재 입력만 기준으로 판단한다. 이전 대화 맥락을 추론하지 않는다.
"""


# 테스트
llm_test(product_classification_prompt, ProductClassificationStructure)

In [10]:
test_df = pd.read_csv("product_classification_label_test_50.csv")

In [11]:
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate

# =========================
# product_classification_node 간단 테스트
# =========================

test_data = [
    ["TV 화면이 너무 어두워", "TVT"],
    ["티비 리모컨 연결 방법 알려줘", "TVT"],
    ["에어컨 자동 청소 기능 어떻게 써?", "ACT"],
    ["제습 잘 되는 에어컨 찾아줘", "ACT"],
    ["냉장고 필터 교체 방법 알려줘", "REF"],
    ["김치냉장고 추천해줘", "REF"],
    ["흡입력 좋은 청소기 찾아줘", "VAC"],
    ["로봇청소기 물걸레 기능 있어?", "VAC"],
    ["세탁기 UE 에러가 뭐야?", "WMT"],
    ["건조 기능 있는 세탁기 찾아줘", "WMT"],

    # 빈 문자열 나와야 하는 케이스
    ["안녕", ""],
    ["오늘 날씨 어때?", ""],
    ["전자레인지 추천해줘", ""],
    ["노트북 찾아줘", ""],
    ["공기청정기 필터 교체 방법 알려줘", ""],
    ["그중에 제일 싼 거", ""],
    ["두 번째 거 설명해줘", ""],
    ["삼성 거 추천해줘", ""],
    ["LG 제품 찾아줘", ""],
    ["건조기 추천해줘", ""],
]

test_df = pd.DataFrame(
    test_data,
    columns=["사용자의 질문", "정답 라벨"]
)

# 프롬프트 생성
product_classification_test_prompt = ChatPromptTemplate.from_messages([
    ("system", product_classification_prompt),
    ("human", "{user_input}")
])

# 체인 생성
product_classification_test_chain = (
    product_classification_test_prompt
    | llm.with_structured_output(ProductClassificationStructure)
)

# 예측 실행
predict_values = []

for _, row in test_df.iterrows():
    result = product_classification_test_chain.invoke({
        "user_input": row["사용자의 질문"]
    })

    predict_values.append(result.product_category)

test_df["predict_label"] = predict_values

# 정답 비교
test_df["is_correct"] = (
    test_df["정답 라벨"].fillna("").astype(str)
    == test_df["predict_label"].fillna("").astype(str)
)

display(test_df)

# 정확도
accuracy = test_df["is_correct"].mean()
print(f"Accuracy: {accuracy:.2%}")

# 오답 확인
wrong_df = test_df[test_df["is_correct"] == False]
display(wrong_df)

,사용자의 질문,정답 라벨,predict_label,is_correct
0,TV 화면이 너무 어두워,TVT,TVT,True
1,티비 리모컨 연결 방법 알려줘,TVT,TVT,True
2,에어컨 자동 청소 기능 어떻게 써?,ACT,ACT,True
3,제습 잘 되는 에어컨 찾아줘,ACT,ACT,True
4,냉장고 필터 교체 방법 알려줘,REF,REF,True
5,김치냉장고 추천해줘,REF,REF,True
6,흡입력 좋은 청소기 찾아줘,VAC,VAC,True
7,로봇청소기 물걸레 기능 있어?,VAC,VAC,True
8,세탁기 UE 에러가 뭐야?,WMT,WMT,True
9,건조 기능 있는 세탁기 찾아줘,WMT,WMT,True


Accuracy: 100.00%


,사용자의 질문,정답 라벨,predict_label,is_correct


In [12]:
test_df.head()

,사용자의 질문,정답 라벨,predict_label,is_correct
0,TV 화면이 너무 어두워,TVT,TVT,True
1,티비 리모컨 연결 방법 알려줘,TVT,TVT,True
2,에어컨 자동 청소 기능 어떻게 써?,ACT,ACT,True
3,제습 잘 되는 에어컨 찾아줘,ACT,ACT,True
4,냉장고 필터 교체 방법 알려줘,REF,REF,True


In [13]:
test_df.to_csv("product_classification_test_result.csv", index=False)

In [23]:
# =========================
# product_field_extraction_schema
# =========================
# 이 스키마들은 제품군별 검색 조건 추출을 위한 Pydantic BaseModel이다.
# 사용자의 자연어 입력을 LLM이 구조화된 검색 필드로 변환할 때 사용한다.
#
# 전체 대화 흐름에서 제품 카테고리는 product_classification_node에서 먼저 판단하고,
# 판단된 product_category 값에 따라 ACTFields, TVTFields, REFFields, VACFields, WMTFields 중
# 하나의 스키마를 선택하여 조건 추출을 수행한다.
#
# 공통 필드는 상품명, 가격, 에너지 소비 효율 등급 조건이며,
# 제품군별로 냉방 능력, 화면 크기, 냉장 용량, 흡입력, 세탁/건조 용량 등
# 해당 제품군에 특화된 검색 조건을 추가로 정의한다.
#
# from_favorites는 사용자 관심/보유 제품 범위 안에서 검색할지 여부를 나타내며,
# vector_search는 사용설명서 기반 RAG 검색에 사용할 짧고 명확한 검색 문장이다.
#
# 이 스키마는 intent router 자체가 아니라,
# intent/product_category 분기 이후 실제 검색 조건을 생성하는
# search condition extraction 단계에서 사용된다.

from typing import Optional, List
from pydantic import BaseModel, Field


class ACTFields(BaseModel):
    from_favorites: bool = Field(
        description="사용자 관심/보유 제품 범위 안에서만 검색해야 하면 True, 전체 제품 범위에서 검색해야 하면 False"
    )

    vector_search: str = Field(
        description="사용설명서 벡터 검색에 사용할 검색 문자열. 사용자의 의도를 유지하되 검색에 적합하게 짧고 명확한 문장으로 변환"
    )

    name_icontains: Optional[str] = Field(
        default=None,
        description="상품명에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    price_gte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이상인 제품 검색"
    )

    price_lte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이하인 제품 검색"
    )

    proficiency_level_gte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이상인 제품 검색"
    )

    proficiency_level_lte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이하인 제품 검색"
    )

    proficiency_level_in: Optional[List[int]] = Field(
        default=None,
        description="에너지 소비 효율 등급 목록 중 하나와 일치하는 제품 검색"
    )

    voltage_gte: Optional[int] = Field(
        default=None,
        description="전압이 지정 값 이상인 제품 검색"
    )

    voltage_lte: Optional[int] = Field(
        default=None,
        description="전압이 지정 값 이하인 제품 검색"
    )

    voltage_in: Optional[List[int]] = Field(
        default=None,
        description="전압 목록 중 하나와 일치하는 제품 검색"
    )

    hertz_gte: Optional[int] = Field(
        default=None,
        description="주파수가 지정 값 이상인 제품 검색"
    )

    hertz_lte: Optional[int] = Field(
        default=None,
        description="주파수가 지정 값 이하인 제품 검색"
    )

    hertz_in: Optional[List[int]] = Field(
        default=None,
        description="주파수 목록 중 하나와 일치하는 제품 검색"
    )

    cool_cap_gte: Optional[int] = Field(
        default=None,
        description="냉방 능력이 지정 값 이상인 제품 검색"
    )

    cool_cap_lte: Optional[int] = Field(
        default=None,
        description="냉방 능력이 지정 값 이하인 제품 검색"
    )

    coverage_gte: Optional[float] = Field(
        default=None,
        description="냉방 면적이 지정 값 이상인 제품 검색"
    )

    coverage_lte: Optional[float] = Field(
        default=None,
        description="냉방 면적이 지정 값 이하인 제품 검색"
    )

    color_icontains: Optional[str] = Field(
        default=None,
        description="색상에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )


class TVTFields(BaseModel):
    name_icontains: Optional[str] = Field(
        default=None,
        description="상품명에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    resol_code_in: Optional[List[str]] = Field(
        default=None,
        description="해상도 코드 목록 중 하나와 일치하는 제품 검색"
    )

    price_gte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이상인 제품 검색"
    )

    price_lte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이하인 제품 검색"
    )

    proficiency_level_gte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이상인 제품 검색"
    )

    proficiency_level_lte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이하인 제품 검색"
    )

    proficiency_level_in: Optional[List[int]] = Field(
        default=None,
        description="에너지 소비 효율 등급 목록 중 하나와 일치하는 제품 검색"
    )

    screen_size_gte: Optional[int] = Field(
        default=None,
        description="화면 크기가 지정 값 이상인 제품 검색"
    )

    screen_size_lte: Optional[int] = Field(
        default=None,
        description="화면 크기가 지정 값 이하인 제품 검색"
    )

    display_type_icontains: Optional[str] = Field(
        default=None,
        description="디스플레이 타입에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    ref_rate_gte: Optional[int] = Field(
        default=None,
        description="주사율이 지정 값 이상인 제품 검색"
    )

    ref_rate_lte: Optional[int] = Field(
        default=None,
        description="주사율이 지정 값 이하인 제품 검색"
    )

    ref_rate_in: Optional[List[int]] = Field(
        default=None,
        description="주사율 목록 중 하나와 일치하는 제품 검색"
    )

    os_type_icontains: Optional[str] = Field(
        default=None,
        description="운영체제 이름에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )


class REFFields(BaseModel):
    name_icontains: Optional[str] = Field(
        default=None,
        description="상품명에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    price_gte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이상인 제품 검색"
    )

    price_lte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이하인 제품 검색"
    )

    proficiency_level_gte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이상인 제품 검색"
    )

    proficiency_level_lte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이하인 제품 검색"
    )

    proficiency_level_in: Optional[List[int]] = Field(
        default=None,
        description="에너지 소비 효율 등급 목록 중 하나와 일치하는 제품 검색"
    )

    install_type_icontains: Optional[str] = Field(
        default=None,
        description="설치 타입에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    door_cnt_gte: Optional[int] = Field(
        default=None,
        description="도어 개수가 지정 값 이상인 제품 검색"
    )

    door_cnt_lte: Optional[int] = Field(
        default=None,
        description="도어 개수가 지정 값 이하인 제품 검색"
    )

    door_cnt_in: Optional[List[int]] = Field(
        default=None,
        description="도어 개수 목록 중 하나와 일치하는 제품 검색"
    )

    total_cap_gte: Optional[int] = Field(
        default=None,
        description="전체 용량이 지정 값 이상인 제품 검색"
    )

    total_cap_lte: Optional[int] = Field(
        default=None,
        description="전체 용량이 지정 값 이하인 제품 검색"
    )

    color_icontains: Optional[str] = Field(
        default=None,
        description="색상에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    door_type_icontains: Optional[str] = Field(
        default=None,
        description="도어 타입에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    cool_type_icontains: Optional[str] = Field(
        default=None,
        description="냉각 방식에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    smart_diag_in: Optional[List[int]] = Field(
        default=None,
        description="스마트 진단 지원 여부 값 목록 중 하나와 일치하는 제품 검색"
    )

    ice_maker_in: Optional[List[int]] = Field(
        default=None,
        description="아이스 메이커 지원 여부 값 목록 중 하나와 일치하는 제품 검색"
    )


class VACFields(BaseModel):
    name_icontains: Optional[str] = Field(
        default=None,
        description="상품명에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    price_gte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이상인 제품 검색"
    )

    price_lte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이하인 제품 검색"
    )

    proficiency_level_gte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이상인 제품 검색"
    )

    proficiency_level_lte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이하인 제품 검색"
    )

    proficiency_level_in: Optional[List[int]] = Field(
        default=None,
        description="에너지 소비 효율 등급 목록 중 하나와 일치하는 제품 검색"
    )

    suction_power_gte: Optional[int] = Field(
        default=None,
        description="흡입력이 지정 값 이상인 제품 검색"
    )

    suction_power_lte: Optional[int] = Field(
        default=None,
        description="흡입력이 지정 값 이하인 제품 검색"
    )

    battery_cnt_gte: Optional[int] = Field(
        default=None,
        description="배터리 개수가 지정 값 이상인 제품 검색"
    )

    battery_cnt_lte: Optional[int] = Field(
        default=None,
        description="배터리 개수가 지정 값 이하인 제품 검색"
    )

    battery_cnt_in: Optional[List[int]] = Field(
        default=None,
        description="배터리 개수 목록 중 하나와 일치하는 제품 검색"
    )

    color_icontains: Optional[str] = Field(
        default=None,
        description="색상에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )


class WMTFields(BaseModel):
    name_icontains: Optional[str] = Field(
        default=None,
        description="상품명에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    price_gte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이상인 제품 검색"
    )

    price_lte: Optional[int] = Field(
        default=None,
        description="가격이 지정 값 이하인 제품 검색"
    )

    proficiency_level_gte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이상인 제품 검색"
    )

    proficiency_level_lte: Optional[int] = Field(
        default=None,
        description="에너지 소비 효율 등급이 지정 값 이하인 제품 검색"
    )

    proficiency_level_in: Optional[List[int]] = Field(
        default=None,
        description="에너지 소비 효율 등급 목록 중 하나와 일치하는 제품 검색"
    )

    washing_cap_gte: Optional[int] = Field(
        default=None,
        description="세탁 용량이 지정 값 이상인 제품 검색"
    )

    washing_cap_lte: Optional[int] = Field(
        default=None,
        description="세탁 용량이 지정 값 이하인 제품 검색"
    )

    drying_cap_gte: Optional[int] = Field(
        default=None,
        description="건조 용량이 지정 값 이상인 제품 검색"
    )

    drying_cap_lte: Optional[int] = Field(
        default=None,
        description="건조 용량이 지정 값 이하인 제품 검색"
    )

    color_icontains: Optional[str] = Field(
        default=None,
        description="색상에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    door_design_icontains: Optional[str] = Field(
        default=None,
        description="도어 디자인에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    control_type_icontains: Optional[str] = Field(
        default=None,
        description="조작 방식에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    door_type_icontains: Optional[str] = Field(
        default=None,
        description="도어 타입에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

    water_temp_icontains: Optional[str] = Field(
        default=None,
        description="지원 물온도 옵션에 특정 문자열이 포함된 제품 검색. 대소문자 구분 없음"
    )

In [44]:
# =========================
# answer_with_result_node
# =========================
# 검색 결과를 바탕으로 사용자에게 최종 답변을 생성하는 노드이다.
# 이전 노드에서 추출한 product_category, vector_search, DB 필터 검색 결과,
# 사용설명서 RAG 검색 결과를 입력으로 받아 자연어 응답을 만든다.
#
# 이 노드는 새로운 검색 조건을 판단하지 않는다.
# 이미 검색된 결과를 해석하고, 사용자 질문에 맞게 요약·추천·안내하는 역할만 수행한다.
#
# 검색 결과가 충분하면 근거 기반으로 답변하고,
# 검색 결과가 부족하면 임의로 지어내지 않고 조건 완화나 추가 질문을 안내한다.

from pydantic import BaseModel, Field


class AnswerWithResultStructure(BaseModel):
    answer: str = Field(
        description="검색 결과를 바탕으로 사용자에게 제공할 최종 답변"
    )

from typing_extensions import TypedDict

from typing_extensions import TypedDict
from typing import Optional, List, Dict, Any


class AgentState(TypedDict, total=False):
    # 사용자 입력
    user_input: str

    # 분기/분류 결과
    is_fall_case: bool
    is_subsequence: bool
    product_category: str

    # 검색 조건
    search_fields: Dict[str, Any]
    vector_search: str
    from_favorites: bool

    # 검색 결과
    search_results: List[Dict[str, Any]]
    db_results: List[Dict[str, Any]]
    manual_results: List[Dict[str, Any]]

    # 최종 응답
    answer: str

In [35]:
answer_with_result_prompt = """
너는 전자제품 사용설명서 기반 Q&A 및 제품 검색 챗봇의 최종 답변 생성기이다.

너의 역할은 이전 단계에서 검색된 결과를 바탕으로 사용자에게 자연스럽고 정확한 최종 답변을 제공하는 것이다.

입력으로는 다음 정보가 제공될 수 있다.

1. 사용자 질문
- 사용자가 실제로 입력한 질문이다.

2. 제품 카테고리
- TVT: TV
- ACT: 에어컨
- REF: 냉장고
- VAC: 청소기
- WMT: 세탁기

3. DB 검색 결과
- 가격, 용량, 크기, 색상, 에너지 등급, 성능 조건 등을 기준으로 검색된 제품 목록이다.

4. 사용설명서 검색 결과
- 제품 사용법, 기능 설명, 오류 해결 방법 등 매뉴얼 기반 검색 결과이다.

답변 원칙:

1. 검색 결과 기반 답변
- 반드시 제공된 검색 결과를 우선 근거로 답변한다.
- 검색 결과에 없는 제품명, 기능, 가격, 수치, 설명을 임의로 만들어내지 않는다.
- 검색 결과가 여러 개일 경우 사용자의 조건에 가장 잘 맞는 결과를 우선적으로 정리한다.

2. 제품 추천/검색 답변
- 사용자가 제품 추천이나 조건 검색을 요청했다면, 적합한 제품을 1~3개 정도로 정리한다.
- 각 제품에 대해 사용자가 요청한 조건과 관련된 핵심 정보만 간결하게 설명한다.
- 가격, 용량, 크기, 색상, 에너지 등급, 성능 정보가 있으면 함께 제시한다.
- 검색 결과가 없으면 조건을 완화하거나 다른 조건으로 다시 검색해볼 수 있다고 안내한다.

3. 사용설명서 Q&A 답변
- 사용자가 사용법, 기능, 오류 해결 방법을 물었다면 사용설명서 검색 결과를 바탕으로 설명한다.
- 단계가 필요한 경우 번호를 사용해 순서대로 설명한다.
- 에러코드나 기능 설명은 사용자가 바로 이해할 수 있게 풀어서 설명한다.
- 매뉴얼 검색 결과가 부족하면 확인 가능한 정보가 부족하다고 말하고, 제품명이나 모델명을 추가로 요청한다.

4. 관심/보유 제품 범위 답변
- from_favorites가 True인 경우 사용자의 관심/보유 제품 범위 안에서 확인한 결과임을 자연스럽게 언급한다.
- from_favorites가 False인 경우 전체 제품 범위에서 검색한 결과로 답변한다.
- 단, 이 정보가 입력으로 제공되지 않으면 따로 언급하지 않는다.

5. 응답 스타일
- 과도하게 장황하게 말하지 않는다.
- 사용자의 질문에 직접 답한다.
- 불확실한 내용은 단정하지 않는다.
- 검색 결과가 부족하면 임의로 보완하지 말고 부족하다고 말한다.
- 마지막에는 필요하면 추가 조건을 제안한다.

출력 규칙:
- answer 필드에만 최종 답변을 작성한다.
"""

In [36]:
from langchain_core.prompts import ChatPromptTemplate


def answer_with_result_node(state):
    """
    검색 결과를 바탕으로 최종 답변을 생성하는 노드
    """

    prompt = ChatPromptTemplate.from_messages([
        ("system", answer_with_result_prompt),
        ("human", """
사용자 질문:
{user_input}

제품 카테고리:
{product_category}

검색 조건:
{search_conditions}

DB 검색 결과:
{db_results}

사용설명서 검색 결과:
{manual_results}

관심/보유 제품 범위 검색 여부:
{from_favorites}
""")
    ])

    chain = prompt | llm.with_structured_output(AnswerWithResultStructure)

    result = chain.invoke({
        "user_input": state.get("user_input", ""),
        "product_category": state.get("product_category", ""),
        "search_conditions": state.get("search_conditions", {}),
        "db_results": state.get("db_results", []),
        "manual_results": state.get("manual_results", []),
        "from_favorites": state.get("from_favorites", None),
    })

    return {
        "answer": result.answer
    }

In [40]:
def answer_with_result_node(state):
    prompt = ChatPromptTemplate.from_messages([
        ("system", answer_with_result_prompt),
        ("human", """
사용자 질문:
{user_input}

검색 결과:
{search_results}
""")
    ])

    chain = prompt | llm.with_structured_output(AnswerWithResultStructure)

    result = chain.invoke({
        "user_input": state.get("user_input", ""),
        "search_results": state.get("search_results", [])
    })

    return {
        "answer": result.answer
    }


In [45]:
test_state = {
    "user_input": "흡입력 좋은 청소기 추천해줘",
    "search_results": [
        {
            "name": "무선청소기 A",
            "price": 299000,
            "suction_power": 200,
            "battery_cnt": 2,
            "color": "화이트"
        },
        {
            "name": "무선청소기 B",
            "price": 199000,
            "suction_power": 150,
            "battery_cnt": 1,
            "color": "블랙"
        }
    ]
}

result = answer_with_result_node(test_state)
print(result)

{'answer': '흡입력이 좋은 청소기를 추천해드리겠습니다. 다음 두 가지 제품을 고려해보세요:\n\n1. **무선청소기 A**  \n   - **가격**: 299,000원  \n   - **흡입력**: 200W  \n   - **배터리 수**: 2개  \n   - **색상**: 화이트  \n\n2. **무선청소기 B**  \n   - **가격**: 199,000원  \n   - **흡입력**: 150W  \n   - **배터리 수**: 1개  \n   - **색상**: 블랙  \n\n무선청소기 A가 더 높은 흡입력을 제공하므로, 성능을 중시하신다면 이 제품을 추천드립니다.'}
